In [ ]:
# ============================================================
# SVM CLASSIFICATION - STUDENT PLACEMENT PREDICTION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)


# ============================================================
# 1. LOAD CSV FILE
# ============================================================

data = pd.read_csv("student_placement.csv")


# ============================================================
# 2. SELECT TWO FEATURES
# ============================================================
# We use two features so that the SVM decision boundary
# can be shown on a 2D graph.

X = data[["study_hours", "technical_score"]]

# Target variable
y = data["placement"]


# ============================================================
# 3. SPLIT DATA INTO TRAINING AND TESTING
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n========== DATA SPLIT ==========")
print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


# ============================================================
# 4. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


# ============================================================
# 5. CREATE SVM MODEL
# ============================================================

model = SVC(
    kernel="linear",
    C=1.0
)


# ============================================================
# 6. TRAIN SVM MODEL
# ============================================================

model.fit(X_train_scaled, y_train)

print("\n========== MODEL TRAINED ==========")
print("Algorithm : Support Vector Machine")
print("Kernel    :", model.kernel)
print("C value   :", model.C)


# ============================================================
# 7. PREDICTION ON TEST DATA
# ============================================================

y_pred = model.predict(X_test_scaled)


# ============================================================
# 8. ACCURACY
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n========== MODEL ACCURACY ==========")
print("Accuracy =", accuracy)
print("Accuracy = {:.2f}%".format(accuracy * 100))


# ============================================================
# 9. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, y_pred)

print("\n========== CONFUSION MATRIX ==========")
print(cm)


# ============================================================
# 10. CONFUSION MATRIX GRAPH
# ============================================================

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Placed", "Placed"]
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.title("SVM - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.tight_layout()
plt.show()


# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)


# ============================================================
# 12. SVM EQUATION
# ============================================================

# SVM equation:
#
# w1*x1 + w2*x2 + b = 0

w_scaled = model.coef_[0]
b_scaled = model.intercept_[0]

print("\n========== SVM EQUATION ==========")

print("Scaled-space equation:")

print(
    "{:.4f} * StudyHours_scaled + "
    "{:.4f} * TechnicalScore_scaled + "
    "{:.4f} = 0".format(
        w_scaled[0],
        w_scaled[1],
        b_scaled
    )
)


# ============================================================
# 13. CONVERT EQUATION TO ORIGINAL FEATURE SCALE
# ============================================================

# StandardScaler equation:
#
# z = (x - mean) / standard deviation
#
# SVM equation:
#
# w1*z1 + w2*z2 + b = 0

w_original = w_scaled / scaler.scale_

b_original = (
    b_scaled
    - np.sum(
        w_scaled *
        scaler.mean_ /
        scaler.scale_
    )
)

print("\nOriginal-feature equation:")

print(
    "{:.4f} * StudyHours + "
    "{:.4f} * TechnicalScore "
    "{:+.4f} = 0".format(
        w_original[0],
        w_original[1],
        b_original
    )
)


# ============================================================
# 14. DECISION BOUNDARY
# ============================================================

# Equation:
#
# w1*x1 + w2*x2 + b = 0
#
# Therefore:
#
# x2 = -(w1*x1 + b) / w2

x_min = X["study_hours"].min() - 0.5

x_max = X["study_hours"].max() + 0.5

x_values = np.linspace(
    x_min,
    x_max,
    200
)

decision_boundary = (
    -(w_original[0] * x_values + b_original)
    / w_original[1]
)


# ============================================================
# 15. MARGIN LINES
# ============================================================

# Positive margin:
#
# w1*x1 + w2*x2 + b = +1
#
# Negative margin:
#
# w1*x1 + w2*x2 + b = -1

margin_positive = (
    -(w_original[0] * x_values + b_original - 1)
    / w_original[1]
)

margin_negative = (
    -(w_original[0] * x_values + b_original + 1)
    / w_original[1]
)


# ============================================================
# 16. SUPPORT VECTORS
# ============================================================

support_vectors_scaled = model.support_vectors_

# Convert support vectors back to original feature scale

support_vectors_original = scaler.inverse_transform(
    support_vectors_scaled
)


# ============================================================
# 17. SVM DECISION BOUNDARY GRAPH
# ============================================================

plt.figure(figsize=(10, 7))


# ------------------------------------------------------------
# Plot Class 0 - Not Placed
# ------------------------------------------------------------

plt.scatter(
    X[y == 0]["study_hours"],
    X[y == 0]["technical_score"],
    marker="o",
    s=80,
    label="Not Placed (0)"
)


# ------------------------------------------------------------
# Plot Class 1 - Placed
# ------------------------------------------------------------

plt.scatter(
    X[y == 1]["study_hours"],
    X[y == 1]["technical_score"],
    marker="^",
    s=80,
    label="Placed (1)"
)


# ------------------------------------------------------------
# Decision Boundary
# ------------------------------------------------------------

plt.plot(
    x_values,
    decision_boundary,
    linewidth=2,
    label="Decision Boundary"
)


# ------------------------------------------------------------
# Positive Margin
# ------------------------------------------------------------

plt.plot(
    x_values,
    margin_positive,
    linestyle="--",
    label="Margin +1"
)


# ------------------------------------------------------------
# Negative Margin
# ------------------------------------------------------------

plt.plot(
    x_values,
    margin_negative,
    linestyle="--",
    label="Margin -1"
)


# ------------------------------------------------------------
# Support Vectors
# ------------------------------------------------------------

plt.scatter(
    support_vectors_original[:, 0],
    support_vectors_original[:, 1],
    s=180,
    facecolors="none",
    edgecolors="black",
    linewidths=2,
    label="Support Vectors"
)


# ============================================================
# 18. GRAPH LABELS
# ============================================================

plt.xlabel("Study Hours")

plt.ylabel("Technical Score")

plt.title(
    "SVM Classification - Student Placement Prediction"
)

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.show()


# ============================================================
# 19. PREDICT A NEW STUDENT
# ============================================================

# Example:
# Study Hours = 7
# Technical Score = 75

new_student = pd.DataFrame(
    [[7, 75]],
    columns=[
        "study_hours",
        "technical_score"
    ]
)


# Scale the new student's data

new_student_scaled = scaler.transform(
    new_student
)


# Predict

prediction = model.predict(
    new_student_scaled
)


# ============================================================
# 20. NEW STUDENT PREDICTION OUTPUT
# ============================================================

print("\n========== NEW STUDENT PREDICTION ==========")

print(
    "Study Hours     :",
    new_student.iloc[0, 0]
)

print(
    "Technical Score :",
    new_student.iloc[0, 1]
)


if prediction[0] == 1:

    print(
        "Prediction      : PLACED"
    )

else:

    print(
        "Prediction      : NOT PLACED"
    )


# ============================================================
# 21. DECISION FUNCTION VALUE
# ============================================================

decision_value = model.decision_function(
    new_student_scaled
)

print(
    "Decision Function: {:.4f}".format(
        decision_value[0]
    )
)

